# ViFinQA CCL Phase 3 V3.4 source-profile constrained smoke

This private GPU kernel evaluates five hash-bound tables with literal financial-statement headings. Qwen3-8B may output only `balance_sheet`, `income_statement`, `cash_flow_statement`, or abstain. The run is diagnostic only: no output may be promoted, certified, repaired, or used for training.

In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json
import subprocess
import sys
import tarfile

WORKING = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')
SOURCE_DIR = WORKING / 'ai_guru_ccl_phase3_graph_v34_statement_smoke_source'
QWEN_RAW_DIR = WORKING / 'ccl_qwen3_8b_statement_profile_v34_smoke_raw'
QWEN_VALIDATED_DIR = WORKING / 'ccl_qwen3_8b_statement_profile_v34_smoke_validated'
ROUTE_ID = 'qwen3_8b_statement_profile_v34_smoke'

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU is required for this CCL smoke. Enable a Kaggle GPU accelerator.')
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
if GPU_VRAM_GIB < 12:
    raise RuntimeError(f'Qwen3-8B 4-bit requires at least 12 GiB VRAM; observed {GPU_VRAM_GIB} GiB.')
print({'gpu': GPU_NAME, 'vram_gib': GPU_VRAM_GIB, 'route_id': ROUTE_ID, 'python': sys.version.split()[0]})


## Required private inputs

Attach exactly two private datasets: the V3.4 source bundle and the V3.4 statement-smoke job dataset. Internet must be enabled only to download the published Qwen checkpoint.

In [ ]:
SOURCE_ARCHIVE_NAME = 'ai_guru_ccl_phase3_source_v1.bundle'
SOURCE_MANIFEST_NAME = 'ai_guru_ccl_phase3_source_v1.manifest.json'
JOB_MANIFEST_NAME = 'bakeoff_job_manifest.json'
REQUESTS_NAME = 'llm_bakeoff_requests_v1.jsonl'
RESPONSE_SCHEMA_NAME = 'llm_proposal_schema_v1.json'
EXPECTED_VALUES = ['balance_sheet', 'income_statement', 'cash_flow_statement']

def exactly_one_input(name: str) -> Path:
    matches = sorted(INPUT_ROOT.rglob(name))
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one Kaggle input named {name!r}; found {matches}')
    return matches[0]

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

SOURCE_ARCHIVE = exactly_one_input(SOURCE_ARCHIVE_NAME)
SOURCE_MANIFEST = exactly_one_input(SOURCE_MANIFEST_NAME)
JOB_MANIFEST = exactly_one_input(JOB_MANIFEST_NAME)
REQUESTS = exactly_one_input(REQUESTS_NAME)
RESPONSE_SCHEMA = exactly_one_input(RESPONSE_SCHEMA_NAME)
source_manifest = json.loads(SOURCE_MANIFEST.read_text(encoding='utf-8'))
if source_manifest.get('protocol') != 'kaggle_ccl_phase3_source_bundle_v1':
    raise RuntimeError('Unexpected CCL source-bundle protocol.')
archive_contract = (source_manifest.get('outputs') or {}).get('archive') or {}
if archive_contract.get('sha256') != sha256_file(SOURCE_ARCHIVE):
    raise RuntimeError('Source bundle archive does not match its manifest SHA-256.')
if SOURCE_DIR.exists():
    raise FileExistsError(f'Refusing to overwrite source directory: {SOURCE_DIR}')
identity = source_manifest.get('source_bundle') or {}
files = identity.get('files') or []
SOURCE_DIR.mkdir(parents=True)
with tarfile.open(SOURCE_ARCHIVE, mode='r:*') as archive:
    expected_names = [entry['path'] for entry in files] + ['SOURCE_BUNDLE.json']
    members = archive.getmembers()
    if [member.name for member in members] != expected_names or not all(member.isfile() for member in members):
        raise RuntimeError('Source archive members do not exactly match the source identity.')
    for entry in files:
        name, expected_sha = entry['path'], entry['sha256']
        if Path(name).is_absolute() or '..' in Path(name).parts:
            raise RuntimeError(f'Unsafe source member path: {name}')
        handle = archive.extractfile(name)
        if handle is None or hashlib.sha256(handle.read()).hexdigest() != expected_sha:
            raise RuntimeError(f'Source member checksum mismatch: {name}')
        handle = archive.extractfile(name)
        destination = SOURCE_DIR / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(handle.read())
job_manifest = json.loads(JOB_MANIFEST.read_text(encoding='utf-8'))
if (job_manifest.get('protocol') != 'vifinqa_ccl_phase3_bakeoff_v1'
        or job_manifest.get('run_status') != 'prepared_phase_3_inference_not_executed'
        or job_manifest.get('training_eligible') is not False
        or job_manifest.get('model_execution_recorded') is not False):
    raise RuntimeError('Job is not a non-promotable prepared CCL Phase 3 job.')
for path in (REQUESTS, RESPONSE_SCHEMA):
    if ((job_manifest.get('outputs') or {}).get(path.name) or {}).get('sha256') != sha256_file(path):
        raise RuntimeError(f'Job output checksum mismatch: {path.name}')
task = job_manifest.get('task_contract') or {}
if (task.get('fields') != ['table_semantics']
        or ((task.get('field_value_constraints') or {}).get('table_semantics') or {}).get('allowed_proposed_values') != EXPECTED_VALUES
        or (task.get('field_relation_constraints') or {}).get('table_semantics') != ['heading_scopes_table']):
    raise RuntimeError('Prepared job does not have the V3.4 constrained statement task contract.')
schema = json.loads(RESPONSE_SCHEMA.read_text(encoding='utf-8'))
if ((schema.get('response_contract') or {}).get('task_contract') != task):
    raise RuntimeError('Response schema task contract does not match the job manifest.')
routes = {str(route.get('route_id')): route for route in job_manifest.get('routes') or [] if isinstance(route, dict)}
route = routes.get(ROUTE_ID)
if not route or not route.get('enabled') or route.get('model_id') != 'Qwen/Qwen3-8B':
    raise RuntimeError('Prepared job does not contain the declared V3.4 Qwen smoke route.')
print({'source_tree_sha256': identity.get('source_tree_sha256'), 'job_request_sha256': sha256_file(REQUESTS), 'request_count': sum(1 for line in REQUESTS.open(encoding='utf-8') if line.strip()), 'task': task})


## P100-compatible 4-bit runtime

The preflight requires a CUDA wheel containing the scheduled GPU architecture before downloading the checkpoint.

In [ ]:
GPU_CAPABILITY = tuple(int(value) for value in torch.cuda.get_device_capability(0))
if GPU_CAPABILITY < (6, 0):
    raise RuntimeError(f'NF4 4-bit requires compute capability >= 6.0; observed {GPU_CAPABILITY}.')
RUNTIME_LOCK = {'torch': 'torch==2.6.0', 'torchvision': 'torchvision==0.21.0', 'torch_index_url': 'https://download.pytorch.org/whl/cu118', 'tokenizers': 'tokenizers==0.21.4', 'huggingface_hub': 'huggingface-hub==0.30.2', 'safetensors': 'safetensors==0.5.3', 'transformers': 'transformers==4.51.3', 'accelerate': 'accelerate==1.7.0', 'bitsandbytes': 'bitsandbytes==0.45.5', 'sentencepiece': 'sentencepiece==0.2.0'}
if GPU_CAPABILITY < (7, 0):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--upgrade', '--force-reinstall', '--index-url', RUNTIME_LOCK['torch_index_url'], RUNTIME_LOCK['torch']], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--upgrade', '--force-reinstall', '--no-deps', '--index-url', RUNTIME_LOCK['torch_index_url'], RUNTIME_LOCK['torchvision']], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--upgrade', '--force-reinstall', '--no-deps', RUNTIME_LOCK['tokenizers'], RUNTIME_LOCK['huggingface_hub'], RUNTIME_LOCK['safetensors'], RUNTIME_LOCK['transformers'], RUNTIME_LOCK['accelerate'], RUNTIME_LOCK['bitsandbytes'], RUNTIME_LOCK['sentencepiece']], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(SOURCE_DIR), '--no-deps'], check=True)
RUNTIME_PROBE = """
import json, torch, torchvision, bitsandbytes
if not torch.cuda.is_available(): raise RuntimeError('CUDA disappeared after runtime resolution.')
capability = tuple(int(value) for value in torch.cuda.get_device_capability(0))
arch_list = list(torch.cuda.get_arch_list()); expected_arch = f'sm_{capability[0]}{capability[1]}'
if capability < (6, 0) or expected_arch not in arch_list: raise RuntimeError(f'Installed PyTorch cannot run NF4 on {expected_arch}: {arch_list}')
if capability < (7, 0) and not torch.__version__.startswith('2.6.0+cu118'): raise RuntimeError(f'P100 PyTorch runtime was overwritten: {torch.__version__}')
if capability < (7, 0) and not torchvision.__version__.startswith('0.21.0+cu118'): raise RuntimeError(f'P100 torchvision runtime is mismatched: {torchvision.__version__}')
torch.empty((1,), device='cuda').zero_()
print(json.dumps({'cuda_available': True, 'torch_version': torch.__version__, 'torchvision_version': torchvision.__version__, 'torch_cuda_version': torch.version.cuda, 'gpu_name': torch.cuda.get_device_name(0), 'gpu_compute_capability': list(capability), 'torch_arch_list': arch_list, 'bitsandbytes_version': bitsandbytes.__version__}))
"""
probe = subprocess.run([sys.executable, '-c', RUNTIME_PROBE], capture_output=True, text=True)
if probe.returncode:
    print(probe.stdout); print(probe.stderr, file=sys.stderr)
    raise RuntimeError(f'CCL model runtime preflight failed with exit code {probe.returncode}.')
MODEL_RUNTIME = json.loads(probe.stdout.strip().splitlines()[-1])
print({'runtime_lock': RUNTIME_LOCK, 'model_runtime': MODEL_RUNTIME})


## Execute and validate

The deterministic validator rejects any response outside the enum or using a non-heading relation. The receipt binds source, job, runtime and validation artifacts.

In [ ]:
subprocess.run([sys.executable, str(SOURCE_DIR / 'scripts/run_ccl_phase3_model.py'), '--job-manifest', str(JOB_MANIFEST), '--requests', str(REQUESTS), '--response-schema', str(RESPONSE_SCHEMA), '--route-id', ROUTE_ID, '--max-new-tokens', '256', '--output-dir', str(QWEN_RAW_DIR)], check=True)
subprocess.run([sys.executable, str(SOURCE_DIR / 'scripts/validate_ccl_phase3_responses.py'), '--job-manifest', str(JOB_MANIFEST), '--requests', str(REQUESTS), '--route-id', ROUTE_ID, '--raw-responses', str(QWEN_RAW_DIR / 'llm_raw_responses_v1.jsonl'), '--output-dir', str(QWEN_VALIDATED_DIR)], check=True)
qwen_execution = json.loads((QWEN_RAW_DIR / 'model_execution_manifest.json').read_text(encoding='utf-8'))
qwen_validation = json.loads((QWEN_VALIDATED_DIR / 'proposal_validation_manifest.json').read_text(encoding='utf-8'))
for manifest in (qwen_execution, qwen_validation):
    if manifest.get('training_eligible') is not False or manifest.get('certification_allowed') is not False:
        raise RuntimeError('GPU receipt violates the non-promotable CCL contract.')
if qwen_execution.get('route', {}).get('route_id') != ROUTE_ID or qwen_validation.get('route_id') != ROUTE_ID:
    raise RuntimeError('Execution or validation route differs from the V3.4 smoke job receipt.')
if qwen_execution.get('runtime') != MODEL_RUNTIME:
    raise RuntimeError('Execution runtime differs from the preflight runtime receipt.')
receipt = {'schema_version': 1, 'protocol': 'vifinqa_ccl_phase3_kaggle_kernel_v1', 'gpu': {'name': GPU_NAME, 'vram_gib': GPU_VRAM_GIB}, 'model_runtime': MODEL_RUNTIME, 'source_tree_sha256': identity['source_tree_sha256'], 'job_manifest_sha256': sha256_file(JOB_MANIFEST), 'qwen_execution_manifest_sha256': sha256_file(QWEN_RAW_DIR / 'model_execution_manifest.json'), 'qwen_validation_manifest_sha256': sha256_file(QWEN_VALIDATED_DIR / 'proposal_validation_manifest.json'), 'task_contract': task, 'training_eligible': False, 'certification_allowed': False}
(WORKING / 'ccl_phase3_kaggle_receipt_v1.json').write_text(json.dumps(receipt, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(receipt, ensure_ascii=False, indent=2))
print('Download raw and validated V3.4 statement-smoke directories plus ccl_phase3_kaggle_receipt_v1.json.')
